<a href="https://colab.research.google.com/github/18217265596/sx/blob/master/LigandMPNN_Colab_Complete_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LigandMPNN → extract.py → ColabFold

生产流程：LigandMPNN 设计 → 提取高分序列 → 指定 de novo 链 → 非 de novo 链按唯一序列查询并复用 MSA → AlphaFold2-Multimer v3（3 个模型）→ 输出按 ipTM 排序的 CSV。若 ipTM 缺失，则使用 pTM 作为排序值。结构文件不会保留。

In [ ]:
# 0. 上传正式生产用 PDB
from google.colab import files
from pathlib import Path
import os, re, sys, json, csv, shutil, hashlib, subprocess

uploaded = files.upload()
pdb_items = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".pdb")]
if len(pdb_items) != 1:
    raise ValueError("请一次只上传一个 .pdb 文件。")

name, data = pdb_items[0]
ROOT = Path("/content/LigandMPNN")
INPUT_DIR = Path("/content/user_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
USER_PDB = INPUT_DIR / Path(name).name
USER_PDB.write_bytes(data)
print("PDB:", USER_PDB)

In [ ]:
# 1. 所有用户可指定参数（LigandMPNN、extract.py、ColabFold）
CHECKPOINT_OPTIONS = {
    1:  ("protein_mpnn", "proteinmpnn_v_48_002.pt", "ProteinMPNN, 0.02 Å noise"),
    2:  ("protein_mpnn", "proteinmpnn_v_48_010.pt", "ProteinMPNN, 0.10 Å noise"),
    3:  ("protein_mpnn", "proteinmpnn_v_48_020.pt", "ProteinMPNN, 0.20 Å noise"),
    4:  ("protein_mpnn", "proteinmpnn_v_48_030.pt", "ProteinMPNN, 0.30 Å noise"),
    5:  ("ligand_mpnn", "ligandmpnn_v_32_005_25.pt", "LigandMPNN, 0.05 Å noise"),
    6:  ("ligand_mpnn", "ligandmpnn_v_32_010_25.pt", "LigandMPNN, 0.10 Å noise"),
    7:  ("ligand_mpnn", "ligandmpnn_v_32_020_25.pt", "LigandMPNN, 0.20 Å noise"),
    8:  ("ligand_mpnn", "ligandmpnn_v_32_030_25.pt", "LigandMPNN, 0.30 Å noise"),
    9:  ("per_residue_label_membrane_mpnn", "per_residue_label_membrane_mpnn_v_48_020.pt", "MembraneMPNN, per-residue labels"),
    10: ("global_label_membrane_mpnn", "global_label_membrane_mpnn_v_48_020.pt", "MembraneMPNN, global label"),
    11: ("soluble_mpnn", "solublempnn_v_48_002.pt", "SolubleMPNN, 0.02 Å noise"),
    12: ("soluble_mpnn", "solublempnn_v_48_010.pt", "SolubleMPNN, 0.10 Å noise"),
    13: ("soluble_mpnn", "solublempnn_v_48_020.pt", "SolubleMPNN, 0.20 Å noise"),
    14: ("soluble_mpnn", "solublempnn_v_48_030.pt", "SolubleMPNN, 0.30 Å noise"),
    15: ("sidechain_packer", "ligandmpnn_sc_v_32_002_16.pt", "仅用于侧链打包"),
}
for number, (_, filename, description) in CHECKPOINT_OPTIONS.items():
    print(f"{number:>2}: {filename:<48} | {description}")

# LigandMPNN
TASK_CHECKPOINT_ID = 6
CHAINS_TO_DESIGN = "A"
SEED = 112
TEMPERATURE = 0.10
BATCH_SIZE = 10
NUMBER_OF_BATCHES = 10
PARSE_ATOMS_WITH_ZERO_OCCUPANCY = 1
SAVE_STATS = 1
FIXED_RESIDUES = ""
REDESIGNED_RESIDUES = ""
VERBOSE = 1
PACK_SIDE_CHAINS = False
NUMBER_OF_PACKS_PER_DESIGN = 1

# extract.py
EXTRACT_TOP_N = 20
EXTRACT_SOURCE_GLOB = "*.fa"
EXTRACT_COMBINED_FASTA_NAME = "all_sequences.fa"
EXTRACT_TSV_NAME = "top_unique_sequences.tsv"

# ColabFold
RUN_COLABFOLD = True
COLABFOLD_MSA_SERVER = "https://api.colabfold.com"
COLABFOLD_USE_ENV = True
COLABFOLD_USE_FILTER = True
COLABFOLD_MODEL_TYPE = "alphafold2_multimer_v3"
COLABFOLD_NUM_RECYCLES = 3
COLABFOLD_NUM_MODELS = 3
COLABFOLD_MODEL_ORDER = [1, 2, 3]
COLABFOLD_NUM_SEEDS = 1
COLABFOLD_USE_DROPOUT = False
COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE = "auto"
COLABFOLD_MAX_MSA = "auto"
COLABFOLD_NUM_RELAX = 0
COLABFOLD_CALC_EXTRA_PTM = True
COLABFOLD_MAX_BINDERS = None
COLABFOLD_KEEP_EXISTING_RESULTS = True
COLABFOLD_JOB_PREFIX = "binder_complex"
COLABFOLD_OUTPUT_DIR_NAME = "colabfold_results"
COLABFOLD_FINAL_CSV_NAME = "colabfold_ranked_iptm_ptm.csv"

# 输出
DOWNLOAD_FINAL_CSV = True
DOWNLOAD_RESULTS_ZIP = True
RESULT_ZIP_NAME = "ligandmpnn_colabfold_results"

if TASK_CHECKPOINT_ID not in range(1, 15):
    raise ValueError("TASK_CHECKPOINT_ID 必须为 1–14。")
if FIXED_RESIDUES and REDESIGNED_RESIDUES:
    raise ValueError("FIXED_RESIDUES 与 REDESIGNED_RESIDUES 不能同时使用。")
if COLABFOLD_NUM_MODELS != 3 or COLABFOLD_MODEL_ORDER != [1, 2, 3]:
    raise ValueError("当前流程固定使用模型 1、2、3。")
if COLABFOLD_NUM_RELAX != 0:
    raise ValueError("当前流程不做 Amber relaxation。")
if not COLABFOLD_CALC_EXTRA_PTM:
    raise ValueError("需要计算额外界面评分，请保持 COLABFOLD_CALC_EXTRA_PTM=True。")

In [ ]:
# 2. 指定 de novo 链：该链不使用 MSA
COLABFOLD_DE_NOVO_CHAIN = "A"

COLABFOLD_DE_NOVO_CHAIN = COLABFOLD_DE_NOVO_CHAIN.strip()
if not COLABFOLD_DE_NOVO_CHAIN or "," in COLABFOLD_DE_NOVO_CHAIN:
    raise ValueError("请填写一条 de novo 链，例如 'A'。")
print("de novo chain:", COLABFOLD_DE_NOVO_CHAIN)

In [ ]:
# 3. 安装 LigandMPNN、下载全部 15 个权重并修复兼容性
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(["git","clone","--depth","1","https://github.com/dauparas/LigandMPNN.git",str(ROOT)], check=True)
subprocess.run([
    sys.executable,"-m","pip","install","-q","--upgrade",
    "ProDy==2.6.1","biopython>=1.81","ml-collections==0.1.1","dm-tree==0.1.8"
], check=True)

MODEL_DIR = ROOT / "model_params"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(["bash", str(ROOT/"get_model_params.sh"), str(MODEL_DIR)], check=True)
weight_files = sorted(MODEL_DIR.glob("*.pt"))
if len(weight_files) != 15:
    raise RuntimeError(f"预期 15 个权重，实际 {len(weight_files)} 个。")

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
run_py = ROOT / "run.py"
source = run_py.read_text()
source = source.replace(
    "torch.load(checkpoint_path, map_location=device)",
    "torch.load(checkpoint_path, map_location=device, weights_only=False)",
).replace(
    "torch.load(args.checkpoint_path_sc, map_location=device)",
    "torch.load(args.checkpoint_path_sc, map_location=device, weights_only=False)",
)
run_py.write_text(source)

aliases = {
    r"\bnp\.int\b":"int", r"\bnp\.float\b":"float", r"\bnp\.bool\b":"bool",
    r"\bnp\.object\b":"object", r"\bnp\.str\b":"str", r"\bnp\.complex\b":"complex",
}
for py_file in ROOT.rglob("*.py"):
    text = py_file.read_text()
    patched = text
    for pattern, replacement in aliases.items():
        patched = re.sub(pattern, replacement, patched)
    if patched != text:
        py_file.write_text(patched)

EXTRACT_PY = ROOT / "extract.py"
subprocess.run([
    "wget","-q",
    "https://raw.githubusercontent.com/18217265596/sx/master/extract.py",
    "-O",str(EXTRACT_PY)
], check=True)
print("LigandMPNN ready:", ROOT)

In [ ]:
# 4. 运行 LigandMPNN
MODEL_TYPE, CHECKPOINT_NAME, _ = CHECKPOINT_OPTIONS[TASK_CHECKPOINT_ID]
CHECKPOINT_PATH = MODEL_DIR / CHECKPOINT_NAME
USER_OUT = ROOT / "outputs" / USER_PDB.stem
shutil.rmtree(USER_OUT, ignore_errors=True)

flags = {
    "protein_mpnn":"--checkpoint_protein_mpnn",
    "ligand_mpnn":"--checkpoint_ligand_mpnn",
    "soluble_mpnn":"--checkpoint_soluble_mpnn",
    "per_residue_label_membrane_mpnn":"--checkpoint_per_residue_label_membrane_mpnn",
    "global_label_membrane_mpnn":"--checkpoint_global_label_membrane_mpnn",
}
cmd = [
    sys.executable,"-u","run.py","--model_type",MODEL_TYPE,
    flags[MODEL_TYPE],str(CHECKPOINT_PATH),
    "--seed",str(SEED),"--pdb_path",str(USER_PDB),"--out_folder",str(USER_OUT),
    "--batch_size",str(BATCH_SIZE),"--number_of_batches",str(NUMBER_OF_BATCHES),
    "--temperature",str(TEMPERATURE),
    "--parse_atoms_with_zero_occupancy",str(PARSE_ATOMS_WITH_ZERO_OCCUPANCY),
    "--save_stats",str(SAVE_STATS),"--verbose",str(VERBOSE),
]
if CHAINS_TO_DESIGN: cmd += ["--chains_to_design",CHAINS_TO_DESIGN]
if FIXED_RESIDUES: cmd += ["--fixed_residues",FIXED_RESIDUES]
if REDESIGNED_RESIDUES: cmd += ["--redesigned_residues",REDESIGNED_RESIDUES]
if PACK_SIDE_CHAINS:
    cmd += [
        "--pack_side_chains","1",
        "--checkpoint_path_sc",str(MODEL_DIR/CHECKPOINT_OPTIONS[15][1]),
        "--number_of_packs_per_design",str(NUMBER_OF_PACKS_PER_DESIGN),
    ]

print(" ".join(cmd))
result = subprocess.run(cmd,cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode:
    raise RuntimeError("LigandMPNN 运行失败。")

In [ ]:
# 5. extract.py：提取 overall_confidence 最高的唯一序列
SEQ_DIR = USER_OUT / "seqs"
fastas = sorted(SEQ_DIR.glob(EXTRACT_SOURCE_GLOB))
if not fastas:
    raise FileNotFoundError("未找到 LigandMPNN FASTA。")

COMBINED_FASTA = USER_OUT / EXTRACT_COMBINED_FASTA_NAME
with COMBINED_FASTA.open("w") as out:
    for fasta in fastas:
        text = fasta.read_text()
        out.write(text)
        if text and not text.endswith("\n"):
            out.write("\n")

EXTRACT_TSV = USER_OUT / EXTRACT_TSV_NAME
result = subprocess.run(
    [sys.executable,str(EXTRACT_PY),"--input",str(COMBINED_FASTA),"--top",str(EXTRACT_TOP_N)],
    text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("extract.py 运行失败。")
EXTRACT_TSV.write_text(result.stdout)
print(result.stdout)

In [ ]:
# 6. 安装 ColabFold
if RUN_COLABFOLD:
    subprocess.run([
        sys.executable,"-m","pip","install","-q","--no-warn-conflicts",
        "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
    ], check=True)
    print("ColabFold installed.")

In [ ]:
# 7. 检查链一致性、复用非 de novo 链 MSA，并生成 custom complex A3M
from collections import OrderedDict
from colabfold.colabfold import run_mmseqs2
from colabfold.input import msa_to_str

def pdb_chain_order(path):
    order, seen = [], set()
    for line in path.read_text(errors="replace").splitlines():
        if line.startswith("ATOM"):
            chain = line[21].strip() or "_"
            if chain not in seen:
                seen.add(chain); order.append(chain)
    return order

CHAIN_ORDER = pdb_chain_order(USER_PDB)
if COLABFOLD_DE_NOVO_CHAIN not in CHAIN_ORDER:
    raise ValueError(f"de novo 链不在 PDB 链中：{CHAIN_ORDER}")

candidates = []
for line in EXTRACT_TSV.read_text().splitlines():
    if not line.strip(): continue
    rank, confidence, record_id, full_sequence = line.split("\t",3)
    parts = full_sequence.split(":")
    if len(parts) != len(CHAIN_ORDER):
        raise ValueError("LigandMPNN FASTA 链数与 PDB 链数不一致。")
    chain_sequences = OrderedDict(zip(CHAIN_ORDER,parts))
    candidates.append({
        "rank":int(rank),"confidence":float(confidence),"record_id":record_id,
        "chains":chain_sequences,
    })
if COLABFOLD_MAX_BINDERS is not None:
    candidates = candidates[:int(COLABFOLD_MAX_BINDERS)]

unique_by_chain = OrderedDict()
for chain in CHAIN_ORDER:
    unique_by_chain[chain] = list(OrderedDict.fromkeys(c["chains"][chain] for c in candidates))

for chain, sequences in unique_by_chain.items():
    if chain == COLABFOLD_DE_NOVO_CHAIN:
        policy = "single_sequence_no_msa"
    elif len(sequences) == 1:
        policy = "one_shared_msa_for_all_candidates"
    else:
        policy = "one_msa_per_unique_sequence"
    print(chain, len(sequences), policy)

MSA_CACHE = USER_OUT / "colabfold_msa_cache"
A3M_DIR = USER_OUT / "colabfold_complex_a3m"
MSA_CACHE.mkdir(exist_ok=True)
A3M_DIR.mkdir(exist_ok=True)

msa_cache = {}
for chain, sequences in unique_by_chain.items():
    if chain == COLABFOLD_DE_NOVO_CHAIN:
        continue
    for seq in sequences:
        key = hashlib.sha1(seq.encode()).hexdigest()[:16]
        cache_file = MSA_CACHE / f"{chain}_{key}.a3m"
        if cache_file.exists() and cache_file.stat().st_size:
            a3m = cache_file.read_text()
        else:
            a3m = run_mmseqs2(
                seq,str(MSA_CACHE/f"mmseqs_{chain}_{key}"),
                use_env=COLABFOLD_USE_ENV,use_filter=COLABFOLD_USE_FILTER,
                use_templates=False,use_pairing=False,host_url=COLABFOLD_MSA_SERVER,
                user_agent="ligandmpnn-colabfold-workflow/1.0",
            )[0]
            cache_file.write_text(a3m)
        msa_cache[(chain,seq)] = a3m

manifest = []
for c in candidates:
    unique_sequences, cardinalities, unique_msas = [], [], []
    for chain, seq in c["chains"].items():
        a3m = f">de_novo_{chain}\n{seq}\n" if chain == COLABFOLD_DE_NOVO_CHAIN else msa_cache[(chain,seq)]
        if seq in unique_sequences:
            cardinalities[unique_sequences.index(seq)] += 1
        else:
            unique_sequences.append(seq); cardinalities.append(1); unique_msas.append(a3m)
    complex_a3m = msa_to_str(unique_msas,None,unique_sequences,cardinalities)
    job = f"{COLABFOLD_JOB_PREFIX}_{c['rank']:03d}_id_{c['record_id']}"
    path = A3M_DIR / f"{job}.a3m"
    path.write_text(complex_a3m)
    manifest.append({
        "jobname":job,
        "sequence":c["chains"][COLABFOLD_DE_NOVO_CHAIN],
        "a3m_path":str(path),
    })
print("Prepared:", len(manifest))

In [ ]:
# 8. 运行 ColabFold，生成唯一最终 CSV，并删除结构文件
if RUN_COLABFOLD:
    import pandas as pd
    from colabfold.download import download_alphafold_params
    from colabfold.batch import get_queries, run, set_model_type

    RESULT_DIR = USER_OUT / COLABFOLD_OUTPUT_DIR_NAME
    RESULT_DIR.mkdir(exist_ok=True)
    queries, is_complex = get_queries(A3M_DIR)
    model_type = set_model_type(is_complex,COLABFOLD_MODEL_TYPE)

    DATA_DIR = Path("/content/colabfold_params")
    DATA_DIR.mkdir(exist_ok=True)
    download_alphafold_params(model_type,DATA_DIR)

    recycle_tol = 0.5 if COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE == "auto" else float(COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE)
    max_msa = None if COLABFOLD_MAX_MSA == "auto" else COLABFOLD_MAX_MSA

    run(
        queries=queries,result_dir=RESULT_DIR,use_templates=False,custom_template_path=None,
        num_relax=0,msa_mode="custom",model_type=model_type,
        num_models=3,num_recycles=int(COLABFOLD_NUM_RECYCLES),
        relax_max_iterations=0,recycle_early_stop_tolerance=recycle_tol,
        num_seeds=COLABFOLD_NUM_SEEDS,use_dropout=COLABFOLD_USE_DROPOUT,
        model_order=[1,2,3],is_complex=True,data_dir=DATA_DIR,
        keep_existing_results=COLABFOLD_KEEP_EXISTING_RESULTS,rank_by="auto",
        pair_mode="unpaired",pairing_strategy="greedy",stop_at_score=100,
        prediction_callback=None,dpi=100,zip_results=False,save_all=False,
        max_msa=max_msa,use_cluster_profile=True,input_features_callback=None,
        save_recycles=False,user_agent="ligandmpnn-colabfold-workflow/1.0",
        calc_extra_ptm=True,
    )

    manifest_by_job = {m["jobname"]:m for m in manifest}
    rows = []
    pattern = re.compile(r"^(?P<job>.+)_scores_rank_(?P<rank>\d+)_(?P<tag>.+)\.json$")
    for score_file in RESULT_DIR.glob("*_scores_*.json"):
        match = pattern.match(score_file.name)
        if not match: continue
        job = match.group("job").removesuffix(".custom")
        if job not in manifest_by_job: continue
        scores = json.loads(score_file.read_text())
        iptm = scores.get("iptm")
        ptm = scores.get("ptm")
        rows.append({
            "job":job,
            "sequence":manifest_by_job[job]["sequence"],
            "iptm":iptm,
            "ptm":ptm,
        })
    if not rows:
        raise RuntimeError("未找到可解析的 ColabFold score JSON。")

    scores_df = pd.DataFrame(rows)
    scores_df["iptm"] = pd.to_numeric(scores_df["iptm"],errors="coerce")
    scores_df["ptm"] = pd.to_numeric(scores_df["ptm"],errors="coerce")
    scores_df["_score"] = scores_df["iptm"].where(
        scores_df["iptm"].notna(), scores_df["ptm"]
    )

    # 每条序列在 3 个模型中先取 ipTM（缺失则 pTM）最高的一条。
    best = (
        scores_df.sort_values(["job","_score"],ascending=[True,False],na_position="last")
        .groupby("job",as_index=False)
        .first()
    )
    # 全部候选按 ipTM 排序；ipTM 缺失时以 pTM 作为排序值。
    best["_sort"] = best["iptm"].where(best["iptm"].notna(),best["ptm"])
    best = best.sort_values("_sort",ascending=False,na_position="last")
    FINAL_CSV = USER_OUT / COLABFOLD_FINAL_CSV_NAME
    best[["sequence","iptm","ptm"]].to_csv(FINAL_CSV,index=False)

    # 用户不需要结构：评分提取完成后删除整个 ColabFold 结构输出目录。
    shutil.rmtree(RESULT_DIR,ignore_errors=True)
    shutil.rmtree(A3M_DIR,ignore_errors=True)
    shutil.rmtree(MSA_CACHE,ignore_errors=True)

    print("Final CSV:", FINAL_CSV)
    display(best[["sequence","iptm","ptm"])

In [ ]:
# 9. 下载最终 CSV；可选下载不含结构文件的完整结果 ZIP
from google.colab import files
if DOWNLOAD_FINAL_CSV:
    files.download(str(FINAL_CSV))

archive = shutil.make_archive(
    f"/content/{RESULT_ZIP_NAME}","zip",root_dir=str(USER_OUT)
)
print("Archive:", archive)
if DOWNLOAD_RESULTS_ZIP:
    files.download(archive)